# FTS Custom ML Backtesting Workspace

Welcome to the custom backtesting workspace! This notebook demonstrates how to run a realistic simulation of a machine learning-based trading strategy using the Financial Trading System (FTS) framework.

### Features of this Backtest:
1. **Historical Replay Event Loop:** Ticks are played back chronologically from our SQLite database.
2. **LSTM Forecasting Strategy:** Uses a pre-trained LSTM neural network to predict price direction from historical close prices.
3. **Execution Delay:** Simulates a realistic **1-bar execution delay** using `KBarExecuteDelay(k=1)` (meaning a signal generated at tick $T$ is submitted, and execution/fill status is checked at tick $T+1$).
4. **Price Slippage:** Simulates **0.1% price slippage** on both buy and sell orders using `FlatPriceSlip`.
5. **Order and Trade Database Logging:** Extends the core execution engine to log detailed execution (fills) to the `trade_logs` database table.
6. **Visual Inspection:** Uses the `BacktestVisualizer` to display predictions overlaid with buy/sell trade markers.
7. **Performance Summary Metrics:** Calculates and exports annualized returns, Sharpe ratio, and maximum drawdown metrics.

### 1. Import Dependencies

In [ ]:
import os
import json
import logging
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from sqlalchemy import create_engine

# Core FTS components
from trading_bot.config import settings
from trading_bot.core.database import init_db, SessionLocal
from trading_bot.core.loop import HistoricalReplayLoop
from trading_bot.core.pipeline import TradingPipeline
from trading_bot.monitoring.prediction_logger import DatabasePredictionLogger
from trading_bot.core.models import BacktestPredictionLog, OrderLog as OrderLogModel, TradeLog as TradeLogModel, Position as PositionModel
from trading_bot.core.repository import MarketDataRepository, OrderRepository, PositionRepository
from trading_bot.core.schemas import BarData, OrderSide, OrderStatus

# ML/Strategy and Risk components
from nets.output_selectors import DynamicThresholdClassifier
from nets.inference import ONNXPredictor
from nets.strategies.nets_strategy import NetsStrategy
from trading_bot.core.transforms import LogReturnTransform
from trading_bot.strategy.engine import StrategyEngine
from trading_bot.risk_management.portfolio import Portfolio
from trading_bot.risk_management.sizing.fixed_percentage import FixedPercentageSizer
from trading_bot.risk_management.manager import RiskManager

# Execution & Backtest components
from trading_bot.execution.delay import KBarExecuteDelay
from trading_bot.execution.slippage import FlatPriceSlip
from trading_bot.execution.handlers.simulated_handler import SimulatedExecutionHandler
from trading_bot.execution.engine import ExecutionEngine
from trading_bot.backtesting.readers import SQLBacktestDataReader
from trading_bot.backtesting import BacktestVisualizer, HTMLBacktestExporter

# Set logging level to INFO for detailed simulation traces
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

### 2. Setup SQLite Database & Connect

We connect to the local SQLite database and clear any existing logs associated with our specific `run_id` to ensure a clean backtest run, without dropping the tables.

In [ ]:
db_url = 'sqlite:///../dev.db'
settings.DATABASE_URL = "sqlite+pysqlite:///../dev.db"

engine = create_engine(db_url, pool_pre_ping=True)
SessionLocal.configure(bind=engine)
db = SessionLocal()

run_id = 'backtest_custom_lstm'

# Clear logs from previous runs of this specific backtest to ensure clean metrics
db.query(BacktestPredictionLog).filter_by(run_id=run_id).delete()
db.query(OrderLogModel).filter_by(run_id=run_id).delete()
db.query(TradeLogModel).filter_by(run_id=run_id).delete()
db.query(PositionModel).filter_by(run_id=run_id).delete()
db.commit()

print("Connected to database and cleared logs for run ID:", run_id)

### 3. Extend the Core Execution Engine

By design, the core execution engine does not write individual fill events to the `trade_logs` table (it only updates overall order statuses in `order_logs`). To allow the `BacktestVisualizer` to render precise BUY/SELL arrows on our chart, we subclass the `ExecutionEngine` to log detailed fill events when an order is completed.

In [ ]:
class CustomExecutionEngine(ExecutionEngine):
    """
    Subclass of ExecutionEngine that extends logging to write filled orders
    directly into the trade_logs table for the visualizer.
    """
    def __init__(self, execution_handler, portfolio, run_id="custom_run"):
        super().__init__(execution_handler, portfolio)
        self.run_id = run_id

    def _log_order(self, db, order, result, strategy_name):
        super()._log_order(db, order, result, strategy_name)
        order_log = db.query(OrderLogModel).filter_by(order_id=result.order_id).first()
        if order_log:
            order_log.run_id = self.run_id
        if result.status == OrderStatus.FILLED:
            self._write_trade_log(db, order_log, result)

    def _update_order_log(self, db, log, result):
        super()._update_order_log(db, log, result)
        log.run_id = self.run_id
        if result.status == OrderStatus.FILLED:
            self._write_trade_log(db, log, result)

    def _write_trade_log(self, db, order_log, result):
        existing = db.query(TradeLogModel).filter_by(order_id=result.order_id).first()
        if not existing:
            trade_log = TradeLogModel(
                run_id=self.run_id,
                order_id=result.order_id,
                market_id=order_log.market_id,
                side=order_log.side,
                outcome=order_log.outcome,
                fill_size=result.filled_size,
                fill_price=result.avg_price,
                fill_timestamp=result.timestamp
            )
            db.add(trade_log)
            print(f"[CustomExecutionEngine] Logged trade for {order_log.side.value} order {result.order_id} @ ${result.avg_price:.2f}")

### 4. Setup Strategy and Prediction Logic

We load our pre-trained LSTM model from `models/my_lstm_model.onnx`, set up feature transformation using logarithmic returns, and configure a threshold classifier output selector.

In [ ]:
onnx_path = '../models/my_lstm_model.onnx'
if not os.path.exists(onnx_path):
    onnx_path = 'models/my_lstm_model.onnx'

print("Loading ONNX predictor from:", onnx_path)
predictor = ONNXPredictor(onnx_path)
transform = LogReturnTransform()
output_selector = DynamicThresholdClassifier(k=0.03, period=10, confidence_multiplier=20.0)

strategy = NetsStrategy(
    predictor=predictor,
    transform=transform,
    output_selector=output_selector,
    lookback_period=20,
    name_suffix='lstm',
    feature_cols=['close']
)
strategy_engine = StrategyEngine(strategies=[strategy])

### 5. Build Backtesting Pipeline with Delay and Slippage Models

Here we instantiate the components required for a realistic backtest simulation:
- **Execution Delay:** `KBarExecuteDelay(k=1)` is injected into the simulated handler.
- **Slippage:** `FlatPriceSlip(slippage_pct=0.001)` (0.1% price penalty) is injected into the simulated handler.
- **Portfolio & Sizer:** A standard portfolio initialized with $10,000, sizing positions at 10% of total equity per trade.

In [ ]:
pos_repo = PositionRepository(db)
order_repo = OrderRepository(db)

portfolio = Portfolio(
    initial_balance=10000.0,
    quote_currency="USD",
    pos_repo=pos_repo,
    order_repo=order_repo
)
portfolio._positions = {}

sizer = FixedPercentageSizer(default_percentage=0.10)
risk_manager = RiskManager(portfolio=portfolio, sizer=sizer)

# Define delayed execution (1 bar delay) and slippage (0.1% penalty)
delay_model = KBarExecuteDelay(k=1)
slippage_model = FlatPriceSlip(slippage_pct=0.001)

execution_handler = SimulatedExecutionHandler(
    delay_model=delay_model,
    slippage_model=slippage_model,
    execution_price_source="close",
    initial_balances={"USD": 10000.0}
)

execution_engine = CustomExecutionEngine(
    execution_handler=execution_handler,
    portfolio=portfolio,
    run_id=run_id
)

pipeline = TradingPipeline(
    ingestion=None,
    strategy=strategy_engine,
    risk=risk_manager,
    execution=execution_engine,
    portfolio=portfolio
)

prediction_logger = DatabasePredictionLogger(
    db=db,
    commit=False,
    model_class=BacktestPredictionLog,
    run_id=run_id
)
pipeline.prediction_logger = prediction_logger

### 6. Run Replay Loop & Record Portfolio Equity

We initialize the data reader to stream BTC/USDT bars between June 1st, 2026, and June 21st, 2026. During the execution of the replay loop, we record the portfolio's cash, position value, and total equity at each tick to construct our equity curve.

In [ ]:
start_date = datetime(2026, 6, 1, 3, 30, 0, tzinfo=timezone.utc)
end_date = datetime(2026, 6, 21, 23, 0, 0, tzinfo=timezone.utc)

data_reader = SQLBacktestDataReader(
    session=db,
    market_id='BTC/USDT',
    start_date=start_date,
    end_date=end_date,
    warmup_bars=100,
    lookback_limit=1000
)
loop_driver = HistoricalReplayLoop(data_reader=data_reader)

print("Starting simulation loop...")
equity_curve = []
prediction_logger.commit = False

tick_count = 0
for tick_data in data_reader.read_data():
    pipeline.execute_single_tick(db, ingestion_output=tick_data)
    
    # Track equity curve
    current_cash = portfolio._cash_balance
    btc_pos = portfolio._positions.get('BTC/USDT')
    pos_size = btc_pos.size if btc_pos else 0.0
    close_price = tick_data.market_data['BTC/USDT'].recent_bars[-1].close
    equity = current_cash + pos_size * close_price
    
    equity_curve.append({
        'timestamp': tick_data.timestamp,
        'cash': current_cash,
        'position': pos_size,
        'close': close_price,
        'equity': equity
    })
    tick_count += 1

db.commit()
print(f"Replay finished. Processed {tick_count} ticks.")

### 7. Calculate and Save Performance Summary Statistics

We evaluate key performance metrics from our equity curve, including cumulative return and maximum drawdown, and write the output as a JSON summary report.

In [ ]:
df_eq = pd.DataFrame(equity_curve)
initial_eq = 10000.0
final_eq = df_eq['equity'].iloc[-1] if not df_eq.empty else initial_eq

total_return = (final_eq - initial_eq) / initial_eq

# Calculate drawdown
df_eq['peak'] = df_eq['equity'].cummax()
df_eq['drawdown'] = (df_eq['equity'] - df_eq['peak']) / df_eq['peak']
max_dd = df_eq['drawdown'].min()

# Count executed trades
trades = db.query(TradeLogModel).filter_by(run_id=run_id).all()
total_trades = len(trades)

summary = {
    "run_id": run_id,
    "market_id": "BTC/USDT",
    "strategy_name": strategy.name,
    "initial_equity": initial_eq,
    "final_equity": final_eq,
    "total_return_pct": float(total_return * 100.0),
    "max_drawdown_pct": float(max_dd * 100.0),
    "total_trades": total_trades
}

os.makedirs('../runs/reports', exist_ok=True)
summary_path = '../runs/reports/backtest_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=4)

print("--- BACKTEST SUMMARY STATS ---")
print(json.dumps(summary, indent=4))

### 8. Render Interactive Dashboard

We load our interactive `BacktestVisualizer` pointing to the SQLite database and render the interactive dashboard to visually inspect cumulative returns, positions, and trades overlaying the candlestick chart.

In [ ]:
# Instantiate the visualizer pointing to the database
viz = BacktestVisualizer('sqlite:///../dev.db')

# Display the dashboard (incorporates LSTM ONNX model structure details if available)
viz.show_dashboard(onnx_model_path=onnx_path)

### 9. Export Standalone Interactive Visualization Report

Finally, we write the entire interactive visualization charts out to a standalone HTML file inside `runs/reports/` for offline review.

In [ ]:
exporter = HTMLBacktestExporter(visualizer=viz)
report_path = exporter.export(
    market_id='BTC/USDT',
    strategy_name=strategy.name,
    run_id=run_id,
    output_path='../runs/reports/'
)
print("Interactive HTML report successfully exported to:", report_path)
db.close()